# 15 — Lagged Chain-Loyalty Robustness Check

**Reviewer comment addressed:** *chain-loyalty & terminology concerns* — specifically the worry that the
measured rise in the chain-loyalty parameter during 2019→2021 is a **contemporaneous mechanical artefact**
(the loyalty regressor and the fitted visit outcome are measured in the same year, so they may be
mechanically coupled rather than reflecting a real behavioural shift).

**What it does.** Re-runs the PSO calibration with the chain-loyalty **input feature lagged by one year**:
the target year's observed visits (`C_Percentage_of_Visits`) are fit using the **prior year's** already-scaled
loyalty value, with every other feature left at the true target-year value. If the loyalty rise survives
lagging, it is not merely a within-year coupling.

Because `R_Percentage_of_Visits_by_brand` is a share bounded exactly in `[0,1]` every year, its normalisation
transform (`R·27+1 → [1,28]`) is identical across years, so lagging requires **no renormalisation** — we
substitute the prior year's scaled value directly.

**Stages:** (0) preflight — confirm every `(A_cbg,B_store)` pair exists in the prior year and R∈[0,1];
(1) lagged calibration for POOLED (5,502 CBGs / 282 stores), DEPARTMENT (452210), and GENERAL MERCHANDISE
(452319) for target years 2019–2021; (2) a **placebo** check on 2019 (lagging 2018→2019 loyalty should be a
near-no-op, since the main analysis found no 2018→2019 shift).

**Reproducibility / safety.** PSO uses a single `np.random.seed(42)` with the original call order; exponent
bounds `[1,15]`. `WRITE_OUTPUTS=False` — recomputes in memory and verifies against saved artefacts in
`outputs/chain_loyalty/` without modifying them.


In [ ]:
import os, numpy as np, pandas as pd, logging
logging.getLogger('pyswarms').setLevel(logging.ERROR)
from pyswarms.single.global_best import GlobalBestPSO
pd.set_option('display.width',160); pd.set_option('display.max_columns',None)

REPO='/Users/mohsenbahrami/Desktop/frontiers_repo'
DATA=os.path.join(REPO,'data/model_inputs')
OUT =os.path.join(REPO,'outputs/chain_loyalty')          # renamed dir (formerly item4_*)
FD  =os.path.join(REPO,'outputs/format_disaggregation')  # holds the format sample files
NORM=28; PSO_RANGE=15; SEED=42
WRITE_OUTPUTS=False
TARGETS=[2019,2020,2021]
PARAM=['H_Area_of_store','R_Percentage_of_Visits_by_brand','J_POI_count_where_store_is',
       'K_POI_diversity_where_store_is','L_Demographic_similarity','G_Distance_between_cbg_and_store']
PRETTY={'H_Area_of_store':'Store area','R_Percentage_of_Visits_by_brand':'Chain loyalty',
        'J_POI_count_where_store_is':'POI count','K_POI_diversity_where_store_is':'POI diversity',
        'L_Demographic_similarity':'Demographic similarity','G_Distance_between_cbg_and_store':'Distance'}
def convert(x,lo,hi): return ((x-lo)/(hi-lo))*(NORM-1)+1 if hi!=lo else pd.Series(1.0,index=x.index)
PASS=[]
def check(name,ok,detail=''):
    PASS.append((name,ok,detail)); print(('PASS ' if ok else '*** FAIL *** ')+name+('  '+detail if detail else ''))

## Stage 0 — Preflight (gate: do not calibrate if this fails)

In [ ]:
# 1) every (A_cbg,B_store) in later year exists in earlier year ; 2) R in [0,1] each year
keys={}
for y in [2018,2019,2020,2021]:
    d=pd.read_csv(os.path.join(DATA,f'table_{y}.csv'),usecols=['A_cbg','B_store'])
    keys[y]=set(d['A_cbg'].astype(str)+'|'+d['B_store'].astype(str))
pf1=True
for e,l in [(2018,2019),(2019,2020),(2020,2021)]:
    rate=len(keys[l]&keys[e])/len(keys[l])*100
    print(f'  {e}->{l}: pair match {rate:.4f}%'); pf1 &= rate>=99.9
pf2=True
for y in [2018,2019,2020,2021]:
    r=pd.read_csv(os.path.join(DATA,f'table_{y}.csv'),usecols=[f'R_Percentage_of_Visits_by_brand_{y}'])[f'R_Percentage_of_Visits_by_brand_{y}']
    print(f'  {y}: R range [{r.min():.4f},{r.max():.4f}]'); pf2 &= (r.min()>=0 and r.max()<=1)
check('preflight 1: 100% pair match', pf1)
check('preflight 2: R in [0,1] all years', pf2)
assert pf1 and pf2, 'PREFLIGHT FAILED — not proceeding to PSO'

## Stage 1 — Lagged calibration (POOLED + both formats, target years 2019–2021)

In [ ]:
OPTIONS={'c1':1.5,'c2':1.5,'w':0.9}; BOUNDS=(np.array([1.]*6),np.array([float(PSO_RANGE)]*6))
def run_cbg(g):
    H,R,J,K,L,G=[g[c].values.astype(float) for c in PARAM]
    actual=g['actual'].values.astype(float)
    if actual.max()==0: return None,None
    am=actual.mean(); asd=actual.std()
    def opt(X):
        out=np.empty(X.shape[0])
        for i in range(X.shape[0]):
            p=X[i]; a=(H**p[0])*(R**p[1])*(J**p[2])*(K**p[3])*(L**p[4])/(G**p[5])
            s=a.sum()
            if s!=0: a=a/s
            bsd=a.std()
            out[i]=1.0 if (asd==0 or bsd==0) else 1.0-np.mean((actual-am)*(a-a.mean()))/(asd*bsd)
        return out
    o=GlobalBestPSO(n_particles=20,dimensions=6,options=OPTIONS,bounds=BOUNDS)
    return o.optimize(opt,iters=10,verbose=False)

def build_pooled_lagged(year):
    prior=year-1
    cols=['A_cbg','B_store',f'C_Percentage_of_Visits_{year}','H_Area_of_store',
          'J_POI_count_where_store_is','K_POI_diversity_where_store_is','L_Demographic_similarity',
          'G_Distance_between_cbg_and_store']
    t=pd.read_csv(os.path.join(DATA,f'table_{year}.csv'),usecols=cols)
    pr=pd.read_csv(os.path.join(DATA,f'table_{prior}.csv'),
                   usecols=['A_cbg','B_store',f'R_Percentage_of_Visits_by_brand_{prior}']).rename(
                   columns={f'R_Percentage_of_Visits_by_brand_{prior}':'R_raw_lag'})
    t=t.merge(pr,on=['A_cbg','B_store'],how='left')
    for a in ['H_Area_of_store','J_POI_count_where_store_is','K_POI_diversity_where_store_is',
              'L_Demographic_similarity','G_Distance_between_cbg_and_store']:
        t[a]=convert(t[a],float(t[a].min()),float(t[a].max()))
    t['R_Percentage_of_Visits_by_brand']=t['R_raw_lag']*(NORM-1)+1   # R in[0,1] -> identical transform
    t['actual']=t[f'C_Percentage_of_Visits_{year}']
    return t[['A_cbg','actual']+PARAM]

def build_format_lagged(sample_file,year):
    s=pd.read_csv(os.path.join(FD,sample_file))
    cur=s[s['year']==year].copy()
    prior=s[s['year']==year-1][['A_cbg','B_store','R_Percentage_of_Visits_by_brand']].rename(
        columns={'R_Percentage_of_Visits_by_brand':'R_lag'})
    cur=cur.merge(prior,on=['A_cbg','B_store'],how='left')
    cur['R_Percentage_of_Visits_by_brand']=cur['R_lag']   # prior-year already-scaled value
    cur['actual']=cur['C_Percentage_of_Visits']
    return cur[['A_cbg','actual']+PARAM]

SAMPLES=[('pooled',None),
         ('452210','sample_452210_department_stores.csv'),
         ('452319','sample_452319_general_merchandise.csv')]

np.random.seed(SEED)                          # single global seed, exactly as original lagged script
lag={}                                        # lag[sample][year] -> DataFrame(cbg,cost,6 params)
for sample,sf in SAMPLES:                     # ORDER MATTERS: pooled, 452210, 452319 ; years 2019..2021
    lag[sample]={}
    for year in TARGETS:
        df = build_pooled_lagged(year) if sf is None else build_format_lagged(sf,year)
        rows=[]
        for c in sorted(df['A_cbg'].unique()):
            cost,v=run_cbg(df[df['A_cbg']==c]); rows.append([c,cost]+(list(v) if v is not None else [np.nan]*6))
        lag[sample][year]=pd.DataFrame(rows,columns=['cbg','cost']+PARAM)
    print(f'{sample}: lagged PSO done for {TARGETS}')

In [ ]:
# --- VERIFY lagged PSO reproduces saved PSO_lagged_*.csv (allclose) ---
for sample,_ in SAMPLES:
    ok=True
    for year in TARGETS:
        saved=pd.read_csv(os.path.join(OUT,f'PSO_lagged_{sample}_{year}.csv')).set_index('cbg').sort_index()
        fresh=lag[sample][year].set_index('cbg').sort_index()
        ok &= np.allclose(fresh[['cost']+PARAM].values, saved[['cost']+PARAM].values, rtol=1e-9, atol=1e-9)
    check(f'{sample} lagged PSO reproduces saved CSVs', ok)

## Stage 1 — Fit and % change under lagging (verification vs `stage2_lagged_results.txt`)

In [ ]:
def pct(a,b): return (b-a)/a*100
fit={}; means={}
for sample,_ in SAMPLES:
    fit[sample]={}; means[sample]={}
    for year in TARGETS:
        r=lag[sample][year]; fit[sample][year]=1-r['cost'].mean(); means[sample][year]=r[PARAM].mean()

EXP_FIT={'pooled':{2019:0.6629,2020:0.1723,2021:0.1016},
         '452210':{2019:0.6962,2020:0.1654,2021:0.1293},
         '452319':{2019:0.6948,2020:0.1916,2021:0.1305}}
print('Lagged mean fit (1-cost):')
for sample,_ in SAMPLES:
    got={y:round(fit[sample][y],4) for y in TARGETS}
    print('  ',sample,got); check(f'{sample} lagged fit matches report', all(got[y]==EXP_FIT[sample][y] for y in TARGETS), str(got))

EXP_PCT={
 ('pooled',2019,2020):{'Store area':76.84,'Chain loyalty':24.82,'POI count':41.07,'POI diversity':15.65,'Demographic similarity':3.75,'Distance':-18.81},
 ('pooled',2019,2021):{'Store area':83.10,'Chain loyalty':40.30,'POI count':65.51,'POI diversity':16.69,'Demographic similarity':3.06,'Distance':-27.54},
 ('452210',2019,2020):{'Store area':98.55,'Chain loyalty':4.10,'POI count':52.72,'POI diversity':4.76,'Demographic similarity':2.44,'Distance':-23.66},
 ('452210',2019,2021):{'Store area':107.11,'Chain loyalty':19.93,'POI count':79.70,'POI diversity':11.23,'Demographic similarity':8.51,'Distance':-22.63},
 ('452319',2019,2020):{'Store area':23.15,'Chain loyalty':-1.68,'POI count':61.97,'POI diversity':26.73,'Demographic similarity':-2.33,'Distance':-24.53},
 ('452319',2019,2021):{'Store area':34.59,'Chain loyalty':-3.04,'POI count':83.87,'POI diversity':14.90,'Demographic similarity':5.56,'Distance':-34.86},
}
print('\nLagged %change vs saved report:')
for (sample,y1,y2),exp in EXP_PCT.items():
    okp=True
    for c in PARAM:
        got=round(pct(means[sample][y1][c],means[sample][y2][c]),2)
        if abs(got-exp[PRETTY[c]])>0.01: okp=False; print(f'  MISMATCH {sample} {y1}->{y2} {PRETTY[c]}: {got} vs {exp[PRETTY[c]]}')
    check(f'{sample} {y1}->{y2} lagged %changes match report', okp)

## Stage 2 — Placebo check (lagged-2019 vs unlagged-2019)

Since the main analysis found no significant 2018→2019 parameter shift, substituting 2018 loyalty for 2019
loyalty when fitting the true 2019 outcome should barely move any parameter. Flag threshold |%diff| > 15%.

In [ ]:
# unlagged-2019 pooled means from the ORIGINAL saved outputs (read-only, deterministic)
o=pd.read_csv(os.path.join(REPO,'data/model_outputs/PSO_2019_6params_NYC_norm_28_PSO_15.csv'))
for c in PARAM: o[c]=pd.to_numeric(o[c],errors='coerce')
pooled_unlag=o[PARAM].mean()
# format unlagged-2019 constants (from Item 3 by-format calibration)
unlag={'pooled':{PRETTY[c]:pooled_unlag[c] for c in PARAM},
 '452210':{'Store area':4.3677,'Chain loyalty':7.1395,'POI count':3.4076,'POI diversity':7.0892,'Demographic similarity':7.8643,'Distance':10.9135},
 '452319':{'Store area':5.6470,'Chain loyalty':7.7368,'POI count':3.6785,'POI diversity':6.5172,'Demographic similarity':7.4362,'Distance':10.4610}}
lagged2019={s:{PRETTY[c]:means[s][2019][c] for c in PARAM} for s,_ in SAMPLES}

print(f"{'Parameter':24s} {'Sample':8s} {'Unlagged':>10s} {'Lagged':>10s} {'%diff':>8s}  flag")
maxabs=0; placebo_rows=[]
for s,_ in SAMPLES:
    for c in PARAM:
        pn=PRETTY[c]; u=unlag[s][pn]; lg=lagged2019[s][pn]; d=(lg-u)/u*100
        maxabs=max(maxabs,abs(d)); placebo_rows.append((pn,s,u,lg,d))
        print(f"{pn:24s} {s:8s} {u:>10.4f} {lg:>10.4f} {d:>7.2f}% {'  >15%!!' if abs(d)>15 else ''}")
check('placebo: NO parameter exceeds 15% |%diff|', maxabs<15, f'max |%diff|={maxabs:.2f}%')
# spot-check a few against saved placebo_check_2019.txt values
EXP_PLACEBO={('Chain loyalty','pooled'):-2.63,('Chain loyalty','452210'):-6.67,('Chain loyalty','452319'):-2.29,
             ('Store area','452210'):-3.95,('POI count','452319'):9.01}
okpl=True
for (pn,s),want in EXP_PLACEBO.items():
    u=unlag[s][pn]; lg=lagged2019[s][pn]; got=round((lg-u)/u*100,2)
    if abs(got-want)>0.02: okpl=False; print(f'  placebo MISMATCH {s} {pn}: {got} vs {want}')
check('placebo %diffs match saved placebo_check_2019.txt', okpl)

## Result

Under lagging, the pooled chain-loyalty increase is **largely preserved** (≈+40% by 2021 vs +44% unlagged),
so it is not a within-year mechanical artefact; department stores drive it while general merchandise stays
≈0. The placebo confirms the lag machinery is a near-no-op in the stable pre-COVID year (all |%diff| < 15%,
chain loyalty within a few percent), validating the design.

In [ ]:
print('='*60); print('NOTEBOOK 15 VERIFICATION SUMMARY'); print('='*60)
nfail=sum(1 for _,ok,_ in PASS if not ok)
for name,ok,_ in PASS: print(('PASS ' if ok else 'FAIL ')+name)
print('-'*60); print(f'{len(PASS)-nfail}/{len(PASS)} checks passed')
assert nfail==0, f'{nfail} verification checks FAILED'